# Filtered & multi-tenant RAG: vector search + SQL `WHERE` in one query

Real retrieval rarely means "just find similar text." You need **semantic relevance under hard constraints** — only this category, under this price, above this rating, belonging to this tenant.

With [Infino](https://pypi.org/project/infino/), the vectors and the metadata live in **one table**. Retrieval is a SQL table function, so you combine vector k-NN with a `WHERE` clause on ordinary columns — no separate metadata store, no per-tenant namespaces to manage. We show it on a real **Amazon product catalog** with genuine `price`, `rating`, `category`, and `store` fields.

## Setup

In [1]:
# %pip install infino pyarrow sentence-transformers datasets

In [2]:
import shutil

import infino
import pyarrow as pa

from _shared.embedding import DIM, METRIC, embed, embed_query, as_vector_column
from _shared.datasets import load_amazon

/Users/pratyushlokhande/InfinoAI/workspace/infino/infino-python/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Load a real product catalog

Each product has a title + description (what we search) plus real metadata: price, average rating, category, and store/brand.

In [3]:
products = load_amazon(n=1200)
print(f"loaded {len(products)} products")
p = products[0]
print(p["title"])
print(f"  price=${p['price']}  rating={p['rating']}  category={p['category']!r}  store={p['store']!r}")

loaded 1200 products
GiGi Professional Multi-Purpose Wax Warmer, with See-Through Cover, White
  price=$25.87  rating=4.6  category='All Beauty'  store='GiGi'


## 2. Index text, vectors, and metadata in one table

The metadata columns are stored and SQL-filterable; `text` is full-text indexed and `emb` is vector indexed — all over the same rows.

In [4]:
DB_DIR = "./amazon_data"
shutil.rmtree(DB_DIR, ignore_errors=True)

db = infino.connect(DB_DIR)
schema = pa.schema([
    pa.field("title", pa.large_utf8(), nullable=False),
    pa.field("text", pa.large_utf8(), nullable=False),
    pa.field("price", pa.float64(), nullable=False),
    pa.field("rating", pa.float64(), nullable=False),
    pa.field("category", pa.large_utf8(), nullable=False),
    pa.field("store", pa.large_utf8(), nullable=False),
    pa.field("emb", pa.list_(pa.float32(), DIM), nullable=False),
])
spec = infino.IndexSpec().fts("text").vector("emb", DIM, n_cent=256, metric=METRIC)
table = db.create_table("products", schema, spec)

embeddings = embed([p["text"] for p in products])


def col(key, ty):
    return pa.array([p[key] for p in products], type=ty)


table.append(pa.record_batch([
    col("title", pa.large_utf8()), col("text", pa.large_utf8()),
    col("price", pa.float64()), col("rating", pa.float64()),
    col("category", pa.large_utf8()), col("store", pa.large_utf8()),
    as_vector_column(embeddings),
], schema=schema))
print("indexed", db.query_sql("SELECT COUNT(*) AS n FROM products").to_pydict()["n"][0], "products")

indexed 1200 products


## 3. Vector search + metadata filter, in one SQL query

`vector_search` returns the nearest candidates; the `JOIN` brings back the product columns; the `WHERE` enforces the business constraints. One statement, one table.

**Tip:** when a filter is selective, ask `vector_search` for more candidates (a larger k) so enough survive the `WHERE` — here we pull 300 nearest, then keep the top 5 that pass.

In [5]:
def run(sql: str) -> dict:
    """Run a query; return {} for an empty result set."""
    res = db.query_sql(sql)
    return res.to_pydict() if res.num_rows else {}


def sql_lit(s: str) -> str:
    return "'" + s.replace("'", "''") + "'"


query = "hydrating moisturizer for dry skin"
qvec = ",".join(str(x) for x in embed_query(query))

filtered = run(
    f"SELECT p.title, p.price, p.rating "
    f"FROM vector_search('products', 'emb', '{qvec}', 300) v "
    f"JOIN products p ON v._id = p._id "
    f"WHERE p.price < 25 AND p.rating >= 4.5 "
    f"ORDER BY v.score ASC LIMIT 5"
)
for title, price, rating in zip(filtered["title"], filtered["price"], filtered["rating"]):
    print(f"${price:<6.2f} {rating}★  {title[:80]}")

$16.02  4.8★  Aquaphor Healing Ointment - Moisturizing Skin Protectant for Dry Cracked Hands, 
$4.71   4.5★  Garnier SkinActive Moisture Bomb The Super Hydrating Smoothing Mask, 1.08 F
$9.79   4.5★  Palmers Cocoa Butter Formula With Vitamin E 24 Hour Moisture (25% Bonus) 125G
$6.28   4.7★  Johnson's Skin Nourishing Moisture Baby Body Wash with Aloe Scent & Vitamin E, H
$16.92  4.6★  CeraVe Hydrating Oil Cleanser 236ml


Same query, different constraints — just change the `WHERE`. No reindexing, no second system.

In [6]:
premium = run(
    f"SELECT p.title, p.price FROM vector_search('products', 'emb', '{qvec}', 300) v "
    f"JOIN products p ON v._id = p._id "
    f"WHERE p.price >= 25 ORDER BY v.score ASC LIMIT 5"
)
print("Premium (price >= $25):")
for title, price in zip(premium.get("title", []), premium.get("price", [])):
    print(f"  ${price:<7.2f} {title[:80]}")

Premium (price >= $25):
  $62.12   Mckesson Moisturizer Mckesson 18 Oz. Pump Bottle (#53-28007-18, Sold Per Case)
  $63.96   Jones Road Miracle Balm Au Naturel - 1.76 oz Citrus Scented Moisturizer for All 
  $79.99   Designer Skin Angel, 20-Ounce Bottle
  $31.44   Colonial Dames Concentrated Vitamin E Moisturizing Cream 42,000 I.U. for Hydrati
  $26.95   Raw African Shea Butter 32 oz. / 2 lbs. Jar (2 pack) Bulk Wholesale 100% Pure Na


## 4. Multi-tenant isolation

Multi-tenant retrieval is just another `WHERE`. Here each catalog section (`category`) is a tenant: the **same** query, scoped to one tenant's rows, never leaks results from another — all from one shared table, no per-tenant index.

In [7]:
n_rows = db.query_sql("SELECT COUNT(*) AS n FROM products").to_pydict()["n"][0]
tenants = db.query_sql(
    "SELECT category, COUNT(*) c FROM products GROUP BY category ORDER BY c DESC LIMIT 2"
).to_pydict()["category"]

for tenant in tenants:
    rows = run(
        f"SELECT p.title, p.category FROM vector_search('products', 'emb', '{qvec}', {n_rows}) v "
        f"JOIN products p ON v._id = p._id WHERE p.category = {sql_lit(tenant)} "
        f"ORDER BY v.score ASC LIMIT 3"
    )
    print(f"\ntenant = {tenant!r}")
    for title in rows.get("title", []):
        print("  -", title[:80])


tenant = 'All Beauty'
  - Aquaphor Healing Ointment - Moisturizing Skin Protectant for Dry Cracked Hands, 
  - Garnier SkinActive Moisture Bomb The Super Hydrating Smoothing Mask, 1.08 F
  - Palmers Cocoa Butter Formula With Vitamin E 24 Hour Moisture (25% Bonus) 125G

tenant = 'Health & Personal Care'
  - Johnson's Skin Nourishing Moisture Baby Body Wash with Aloe Scent & Vitamin E, H
  - Aquaphor Baby Healing Ointment Advanced Therapy Skin Protectant, Dry Skin and Di
  - hydraSense Nasal Aspirator Starter Kit


## What just happened

Vector relevance and scalar constraints (price, rating, category, tenant) resolved together in a single SQL query over one Infino table. There was no separate metadata database to keep in sync and no namespace plumbing for tenants — the filter is just a column.

**Next:** [`chat_app/`](chat_app/) ties retrieval, hybrid search, and filters into an end-user Streamlit chatbot over your documents.